In [ ]:
import pandas as pd
import numpy as np
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

# Fungsi sakti untuk hemat RAM
def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1: Setup Selesai!")

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

# Pastikan Battery_Level di Train bersih (hapus teks jika ada, jadikan float)
def fix_battery(val):
    if pd.isna(val): return np.nan
    s = str(val).replace('%', '')
    try: return float(s)
    except: return np.nan

df_train['Battery_Level'] = df_train['Battery_Level'].apply(fix_battery)
df_test['Battery_Level'] = df_test['Battery_Level'].apply(fix_battery)

df_train = reduce_mem(df_train)
df_test = reduce_mem(df_test)

print("Cell 2: Data dimuat dan dibersihkan!")

In [ ]:
def build_expert_features(df):
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # Fitur andalanmu
    df['price_dist_ratio'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.5)
    df['is_suspicious_short'] = ((df['Distance_KM'] < 1.0) & (df['Est_Price_IDR'] > 30000)).astype(int)
    
    # Sensor
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    df['gyro_abs'] = df['Gyro_Z'].abs()
    
    # Device
    df['Device_Brand'] = df['Device_FP'].astype(str).apply(lambda x: x.split('-')[0])
    
    # Efisiensi (Kunci Navigation Issue)
    df['lat_diff'] = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    df['long_diff'] = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    df['euclidean_dist'] = np.sqrt(df['lat_diff']**2 + df['long_diff']**2)
    df['route_efficiency'] = df['Distance_KM'] / (df['euclidean_dist'] + 0.001)

    drop_cols = ['Timestamp', 'Trip_ID', 'Device_FP'] if 'Trip_ID' in df.columns else ['Timestamp', 'Device_FP']
    return df.drop(columns=drop_cols)

y = df_train['Trip_Label']
X = build_expert_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_expert_features(df_test)
print("Cell 3: Expert Features Ready!")

In [ ]:
import gc
# 1. Split murni dulu (Gunakan 15% data untuk hemat RAM)
X_samp, X_val, y_samp, y_val = train_test_split(X, y, train_size=0.15, stratify=y, random_state=42)

# 2. Oversampling FISIK hanya pada X_samp (Training set)
min_mask = y_samp.isin(['Navigation_Issue', 'Service_Complaint', 'Fraud_Indication'])
X_min = X_samp[min_mask]
y_min = y_samp[min_mask]

# Gabungkan: Training sekarang punya data minoritas lebih banyak
X_train = pd.concat([X_samp, X_min, X_min], axis=0) 
y_train = pd.concat([y_samp, y_min, y_min], axis=0)

del X, y, X_samp, y_samp, X_min, y_min
gc.collect()

# 3. Handle Categories
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
X_train[cat_features] = X_train[cat_features].fillna("MISSING").astype(str)
X_val[cat_features] = X_val[cat_features].fillna("MISSING").astype(str)
X_test_final[cat_features] = X_test_final[cat_features].fillna("MISSING").astype(str)

print(f"Cell 4: Train {len(X_train)} baris. Validasi bersih dari duplikasi!")

In [ ]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=5,      # Regularisasi kuat biar gak overfit
    random_strength=2,  # Tambah variasi pohon
    rsm=0.8,            # Pilih 80% fitur per split biar gak fokus ke satu fitur aja
    task_type="CPU",
    loss_function='MultiClass',
    eval_metric='TotalF1', # Fokus langsung ke F1
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

In [ ]:
y_val_pred = model.predict(X_val).flatten()
print("HASIL EVALUASI JUJUR (BUKAN OVERFIT):")
print(f"Macro F1 Score: {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print("-" * 50)
print(classification_report(y_val, y_val_pred))

In [ ]:
test_preds = model.predict(X_test_final).flatten()
submission = pd.DataFrame({"Trip_ID": df_test["Trip_ID"], "Trip_Label": test_preds})
submission.to_csv(os.path.join(SUB_PATH, "submission_expert_final.csv"), index=False)
print("Selesai! Submit file expert ini.")